# Passo 02 — evidências para o schema normalizado v3

Este caderno propõe o contrato de G02 a partir do inventário
integral aprovado `raw-metadata-full-20260724t184418z`.

Ele pode:

- conferir os artefatos de G01 e o fingerprint atual;
- reler metadados raw em streaming;
- produzir livro de campos, conflitos, rejeições, aliases,
  amostras estruturais e pacotes GPT;
- executar, depois de gates humanos, um piloto pareado GPT-5.6.

Ele **não normaliza registros**, não altera `raw/`, não funde nem
descarta campos e não interpreta marcadores, oradores ou turnos.
Todas as flags operacionais nascem desligadas.

In [ ]:
MONTAR_DRIVE = False

if MONTAR_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    print("Drive montado; nenhuma leitura integral foi autorizada.")
else:
    print("Drive não montado. Altere MONTAR_DRIVE para True no Colab.")

## 1. Carregar uma revisão identificável

O Drive é montado antes de qualquer clone, instalação ou import do
projeto. A implementação exige o SDK oficial da OpenAI, mas só o
importa na célula de piloto autorizada.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_REF = "main"
IN_COLAB = Path("/content").is_dir()
REPO_DIR = Path("/content/falando_nela") if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not (REPO_DIR / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                REPO_REF,
                "--single-branch",
                REPO_URL,
                str(REPO_DIR),
            ],
            check=True,
        )
    else:
        dirty = subprocess.check_output(
            ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
            text=True,
        ).strip()
        assert not dirty, f"Clone efêmero com alterações locais: {dirty}"
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "switch", REPO_REF],
            check=True,
        )
        subprocess.run(
            [
                "git",
                "-C",
                str(REPO_DIR),
                "pull",
                "--ff-only",
                "origin",
                REPO_REF,
            ],
            check=True,
        )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(REPO_DIR / "requirements.txt"),
        ],
        check=True,
    )

required = [
    REPO_DIR / "pipeline_dados_v3" / "schema_normalizado.py",
    REPO_DIR
    / "specs"
    / "pipeline_dados_v3"
    / "02_schema_normalizado"
    / "requirements.md",
]
for path in required:
    assert path.exists(), f"Revisão incompleta: {path}"

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
print("Commit carregado:", REPO_COMMIT)

## 2. Fixar entradas e saídas

Informe a localização explícita dos sete artefatos de G01. A saída
de G02 fica em `/content`, fora do Drive. Arquivos opcionais de
revisão também devem ser declarativos: nenhum papel semântico é
inferido pelo código. O SHA-256 aprovado do `manifest.json` já está
fixado na implementação; ele autentica o sétimo artefato, enquanto
o próprio manifest autentica os outros seis.

In [ ]:
from pipeline_dados_v3.schema_normalizado import (
    APPROVED_INVENTORY_MANIFEST_SHA256,
    APPROVED_INVENTORY_OPERATION_ID,
    DEFAULT_OUTPUT_BASE,
    DEFAULT_RAW_ROOT,
    SchemaConfig,
    initialize_field_review,
    prepare_schema_evidence,
    read_jsonl,
    validate_inventory,
)

RAW_ROOT = Path("/content/drive/MyDrive/falando_nela/data/raw")
INVENTORY_ROOT = Path(
    "/content/drive/MyDrive/falando_nela/"
    "auditoria/pipeline_dados_v3/g01"
) / APPROVED_INVENTORY_OPERATION_ID
OUTPUT_BASE = Path("/content/falando_nela_v3_schema")
OPERATION_ID = "schema-evidence-pilot-20260724"
EXPECTED_INVENTORY_MANIFEST_SHA256 = (
    APPROVED_INVENTORY_MANIFEST_SHA256
)

FIELD_REVIEW_PATH = Path("/content/revisao_campos_g02.csv")
MANUAL_ALIASES_PATH = None
API_CATEGORIES_PATH = None

assert RAW_ROOT == DEFAULT_RAW_ROOT
assert OUTPUT_BASE == DEFAULT_OUTPUT_BASE
assert RAW_ROOT not in OUTPUT_BASE.parents

schema_config = SchemaConfig(
    raw_root=RAW_ROOT,
    inventory_root=INVENTORY_ROOT,
    output_base=OUTPUT_BASE,
    operation_id=OPERATION_ID,
    code_commit=REPO_COMMIT,
    expected_inventory_manifest_sha256=(
        EXPECTED_INVENTORY_MANIFEST_SHA256
    ),
    field_review_path=FIELD_REVIEW_PATH,
    manual_alias_path=MANUAL_ALIASES_PATH,
    api_categories_path=API_CATEGORIES_PATH,
    progress_every_files=50,
)
print("Inventário aprovado:", INVENTORY_ROOT)
print("Saída temporária:", schema_config.operation_root)

## 3. Validar G01 sem preparar G02

Esta célula recalcula hashes e confere os totais vinculantes. Ela
também calcula o fingerprint do raw na etapa de preparação; uma
divergência bloqueia a execução.

In [ ]:
VALIDAR_G01 = False
CONFIRMAR_INVENTORY_OPERATION_ID = ""

inventory_manifest = None
if VALIDAR_G01:
    assert MONTAR_DRIVE, "Monte o Drive antes de validar G01."
    assert (
        CONFIRMAR_INVENTORY_OPERATION_ID
        == APPROVED_INVENTORY_OPERATION_ID
    ), "Copie literalmente o operation_id aprovado."
    inventory_manifest, inventory_fields, inventory_issues = (
        validate_inventory(schema_config)
    )
    print("G01 conferido:", inventory_manifest["operation_id"])
    print("Caminhos:", len(inventory_fields))
    print("Inconsistências:", len(inventory_issues))
else:
    print("Validação de G01 bloqueada.")

## 4. Inicializar e revisar o livro de campos

Esta etapa não abre registros raw. Ela cria, a partir do inventário,
uma linha para cada caminho observado. Classifique explicitamente
`semantic_role` como `metadata`, `text`, `technical_control` ou
`unknown`; mantenha decisões humanas e justificativas no mesmo CSV.
O arquivo nunca é sobrescrito automaticamente.

In [ ]:
GERAR_TEMPLATE_REVISAO = False

if GERAR_TEMPLATE_REVISAO:
    assert VALIDAR_G01, "Valide G01 primeiro."
    template_path = initialize_field_review(
        schema_config,
        FIELD_REVIEW_PATH,
    )
    print("Template criado:", template_path)
    print(
        "Revise semantic_role e decision antes da releitura integral."
    )
else:
    print("Geração do template bloqueada.")

## 5. Preparar somente evidências de G02

A releitura é integral e somente leitura. Strings de campos que
permaneçam `unknown` ficam redigidas nas amostras estruturais;
previews só são gerados para campos classificados como `text`.

In [ ]:
PREPARAR_EVIDENCIAS = False
CONFIRMAR_SCHEMA_OPERATION_ID = ""
schema_result = None

if PREPARAR_EVIDENCIAS:
    assert MONTAR_DRIVE, "Monte o Drive antes da releitura integral."
    assert VALIDAR_G01, "Valide G01 primeiro."
    assert FIELD_REVIEW_PATH.is_file(), (
        "Crie e revise o livro de campos antes da releitura."
    )
    assert CONFIRMAR_SCHEMA_OPERATION_ID == OPERATION_ID, (
        "Copie literalmente OPERATION_ID para confirmar."
    )
    schema_result = prepare_schema_evidence(schema_config)
    print("Evidências preparadas:", schema_result["paths"]["report"])
    print(
        "Gate científico:",
        schema_result["manifest"]["scientific_gate"],
    )
else:
    print("Preparação bloqueada; nenhum registro raw foi relido.")

## 6. Revisar os artefatos

O livro nasce com `semantic_role=unknown` e
`decision=nao_avaliado`. Exporte uma cópia revisada para uma nova
operação; não edite o raw. Previews nascem não aprovados.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

if schema_result is None:
    print("Evidências ainda não preparadas.")
else:
    paths = schema_result["paths"]
    display(Markdown(paths["report"].read_text(encoding="utf-8")))
    display(pd.read_csv(paths["field_book"]).head(30))
    display(pd.read_csv(paths["aliases"]).head(30))
    packet_rows = read_jsonl(paths["gpt_packets"])
    display(
        pd.DataFrame(
            [
                {
                    "packet_id": row["packet_id"],
                    "source": row["source"],
                    "dataset": row["dataset"],
                    "record_type": row["record_type"],
                    "chunk_index": row["chunk_index"],
                    "field_count": len(row["structural_evidence"]),
                    "alias_count": len(row["alias_metrics"]),
                }
                for row in packet_rows
            ]
        )
    )
    previews_df = pd.read_json(paths["previews"], lines=True)
    if previews_df.empty:
        print(
            "Sem previews: classifique campos textuais numa revisão "
            "e execute uma nova operação."
        )
    else:
        display(
            previews_df[
                [
                    "context_id",
                    "source",
                    "dataset",
                    "record_type",
                    "field_path",
                    "full_length",
                    "preview",
                    "approved_for_gpt",
                ]
            ]
        )

## 7. Aprovar previews individualmente

Liste apenas IDs já lidos pelo pesquisador. A célula limita cada
trecho a 500 caracteres, mantém `context_only=true` e registra
responsável, data e justificativa. A aprovação não transforma o
preview em evidência estrutural.

In [ ]:
from datetime import date

from pipeline_dados_v3.schema_normalizado import (
    read_jsonl,
    write_jsonl,
)

APROVAR_PREVIEWS = False
CONTEXT_IDS_APROVADOS = []
RESPONSAVEL_PREVIEWS = ""
JUSTIFICATIVA_PREVIEWS = ""

if APROVAR_PREVIEWS:
    assert schema_result is not None, "Prepare as evidências primeiro."
    assert CONTEXT_IDS_APROVADOS, "Liste os context_id aprovados."
    assert RESPONSAVEL_PREVIEWS.strip(), "Informe o responsável."
    assert JUSTIFICATIVA_PREVIEWS.strip(), "Informe a justificativa."
    preview_path = schema_result["paths"]["previews"]
    preview_rows = read_jsonl(preview_path)
    known = {row["context_id"] for row in preview_rows}
    requested = set(CONTEXT_IDS_APROVADOS)
    assert requested <= known, f"IDs desconhecidos: {requested - known}"
    for row in preview_rows:
        if row["context_id"] in requested:
            assert row["context_only"] is True
            assert row["end"] - row["start"] <= 500
            row["approved_for_gpt"] = True
            row["approval_by"] = RESPONSAVEL_PREVIEWS
            row["approval_at"] = date.today().isoformat()
            row["approval_rationale"] = JUSTIFICATIVA_PREVIEWS
    write_jsonl(preview_path, preview_rows)
    print("Previews aprovados:", len(requested))
else:
    print("Aprovação de previews bloqueada.")

## 8. Piloto pareado GPT-5.6

A chave é reutilizada somente de `google.colab.userdata`; ela não é
exibida nem gravada. Informe um JSON de preços versionado para que
custo seja calculado. Cada pacote executa A sem previews e B apenas
com previews aprovados. Não há fallback nem aplicação automática.

In [ ]:
EXECUTAR_PILOTO_GPT = False
CONFIRMAR_PILOTO_OPERATION_ID = ""
PRICING_JSON = Path("/content/gpt-5.6-pricing.json")
PILOT_PACKET_IDS = []
pilot_result = None

if EXECUTAR_PILOTO_GPT:
    assert APROVAR_PREVIEWS, "Aprove previews antes da condição B."
    assert PILOT_PACKET_IDS, (
        "Selecione packet_ids estratificados depois da revisão."
    )
    assert (
        CONFIRMAR_PILOTO_OPERATION_ID == OPERATION_ID
    ), "Copie literalmente OPERATION_ID para confirmar o piloto."
    assert PRICING_JSON.is_file(), "Forneça a tabela de preços."
    if IN_COLAB:
        from google.colab import userdata

        api_key = userdata.get("OPENAI_API_KEY")
        assert api_key, "Cadastre OPENAI_API_KEY nos Secrets do Colab."
        os.environ["OPENAI_API_KEY"] = api_key

    from pipeline_dados_v3.schema_normalizado import run_gpt_pilot

    pilot_result = run_gpt_pilot(
        schema_config.operation_root,
        confirm_operation_id=CONFIRMAR_PILOTO_OPERATION_ID,
        execute_gpt=True,
        pricing_path=PRICING_JSON,
        pilot_packet_ids=set(PILOT_PACKET_IDS),
        model="gpt-5.6",
        reasoning_effort="medium",
    )
    print("Chamadas:", len(pilot_result["execution_rows"]))
    print("Gate:", pilot_result["manifest"]["scientific_gate"])
else:
    print("Piloto GPT bloqueado; nenhuma chamada paga foi feita.")

## 9. Avaliar A/B depois da revisão cega

A avaliação requer CSV humano com `pair_id`, `condition`,
`proposal_id`, `accepted`, `unsupported_category`,
`incorrect_alias` e `insufficient_evidence`. G02 continua pendente
depois do cálculo.

In [ ]:
AVALIAR_AB = False
REVIEW_CSV = Path("/content/revisao_propostas_gpt.csv")
DECISAO_HUMANA_PREVIEWS = ""

if AVALIAR_AB:
    assert pilot_result is not None, "Execute e revise o piloto primeiro."
    assert REVIEW_CSV.is_file(), "Forneça o CSV de revisão humana."
    from pipeline_dados_v3.schema_normalizado import (
        evaluate_context_ab,
    )

    ab_rows = evaluate_context_ab(
        schema_config.operation_root,
        review_path=REVIEW_CSV,
        human_preview_decision=DECISAO_HUMANA_PREVIEWS,
    )
    display(pd.DataFrame(ab_rows))
    print("Avaliação pronta; G02 ainda exige decisão humana.")
else:
    print("Avaliação A/B bloqueada.")

## 10. Gerar o catálogo global compacto sem reler o raw

Esta continuação reutiliza `inventory_manifest`, `inventory_fields` e
`inventory_issues` já validados. Ela lê somente as amostras seguras de
G01, representa os 23.786 caminhos uma vez cada e cria um arquivo `.txt`
para o modelo, além do crosswalk integral. Reexecutar a célula reutiliza
artefatos idênticos; um arquivo divergente nunca é sobrescrito. O perfil
`schema_core` mantém no TXT os caminhos, tipos, cobertura, estados,
cardinalidade, tamanho máximo e conflitos; todas as métricas integrais
continuam preservadas no crosswalk.

In [ ]:
GERAR_CATALOGO_GLOBAL = False
GLOBAL_CATALOG_DIR = Path("/content/falando_nela_g02_global_core")
global_catalog_result = None

if GERAR_CATALOGO_GLOBAL:
    assert inventory_manifest is not None, "Valide G01 primeiro."
    assert len(inventory_fields) == 23_786
    from pipeline_dados_v3.schema_normalizado import (
        build_compact_global_catalog,
    )

    inventory_samples = read_jsonl(
        INVENTORY_ROOT / "amostras_campos.jsonl"
    )
    global_catalog_result = build_compact_global_catalog(
        inventory_manifest=inventory_manifest,
        inventory_manifest_sha256=(
            EXPECTED_INVENTORY_MANIFEST_SHA256
        ),
        field_rows=inventory_fields,
        issue_rows=inventory_issues,
        sample_rows=inventory_samples,
        output_dir=GLOBAL_CATALOG_DIR,
        catalog_profile="schema_core",
        standard_sample_fields=1,
        ccj_sample_fields=4,
        samples_per_field=1,
    )
    global_manifest = global_catalog_result["manifest"]
    global_paths = global_catalog_result["paths"]
    print("Catálogo:", global_paths["catalog"])
    print("Bytes:", global_paths["catalog"].stat().st_size)
    print("Caminhos:", global_manifest["counts"]["field_paths"])
    print("Conflitos:", global_manifest["counts"]["type_conflicts"])
    print(
        "senado/ccj_notas:",
        global_manifest["counts"]["ccj_notas_field_paths"],
    )
    print("Amostras seguras:", global_manifest["counts"]["safe_sample_rows"])
else:
    print("Geração do catálogo global bloqueada.")

## 11. Enviar o arquivo e contar tokens exatamente

Esta célula faz upload somente de `catalogo_global_gpt56.txt` com
`purpose=user_data` e usa o contador oficial da Responses API com o prompt
completo. Ela não pede uma resposta ao GPT e não propõe nem aplica schema.
O `file_id` e a contagem ficam registrados para reutilização na chamada
global posterior.

In [ ]:
import json

ENVIAR_E_CONTAR_CATALOGO_GLOBAL = False
global_file_id = None
global_input_tokens = None
global_model_input = None

if ENVIAR_E_CONTAR_CATALOGO_GLOBAL:
    assert global_catalog_result is not None, "Gere o catálogo primeiro."
    if IN_COLAB:
        from google.colab import userdata

        api_key = userdata.get("OPENAI_API_KEY")
    else:
        api_key = os.environ.get("OPENAI_API_KEY")
    assert api_key, "OPENAI_API_KEY não está disponível."

    from openai import OpenAI
    from pipeline_dados_v3.schema_normalizado import (
        GPT56_MAX_INPUT_TOKENS,
        global_schema_prompt,
        sha256_file,
    )

    client = OpenAI(api_key=api_key)
    catalog_path = global_catalog_result["paths"]["catalog"]
    with catalog_path.open("rb") as handle:
        uploaded = client.files.create(
            file=handle,
            purpose="user_data",
        )
    global_file_id = uploaded.id
    global_model_input = [
        {
            "role": "user",
            "content": [
                {"type": "input_file", "file_id": global_file_id},
                {"type": "input_text", "text": global_schema_prompt()},
            ],
        }
    ]
    token_count = client.responses.input_tokens.count(
        model="gpt-5.6",
        input=global_model_input,
    )
    global_input_tokens = token_count.input_tokens
    receipt = {
        "model": "gpt-5.6",
        "file_id": global_file_id,
        "catalog_sha256": sha256_file(catalog_path),
        "input_tokens": global_input_tokens,
        "max_input_tokens": GPT56_MAX_INPUT_TOKENS,
        "fits": global_input_tokens <= GPT56_MAX_INPUT_TOKENS,
    }
    receipt_path = GLOBAL_CATALOG_DIR / "upload_token_count.json"
    receipt_path.write_text(
        json.dumps(receipt, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print("file_id:", global_file_id)
    print("Tokens exatos:", global_input_tokens)
    print("Limite de entrada:", GPT56_MAX_INPUT_TOKENS)
    print("Cabe em uma chamada:", receipt["fits"])
    print("Recibo:", receipt_path)
else:
    print("Upload e contagem bloqueados; nenhuma chamada foi feita.")

## 12. Submeter uma única proposta global em background

Esta continuação reutiliza diretamente o catálogo `schema_core` e o `file_id`
já criados: não refaz G01, não relê o raw e não exige executar novamente as
células anteriores. Antes da chamada, ela copia catálogo, crosswalk, amostras,
manifest e recibo de tokens para o Drive. A contagem é repetida com o JSON
Schema exato da geração. Se já houver recibo de submissão compatível no Drive,
o `response_id` será reutilizado e uma segunda chamada paga não será criada.

Com 691.302 tokens medidos antes de acrescentar o contrato de saída e limite de
32.000 tokens de saída, a faixa conservadora estimada é US$ 8,35–10,10 no
preço standard de contexto longo vigente em 2026-07-24. A célula revalida o
limite e registra a nova contagem e o teto junto ao recibo da submissão.

In [ ]:
import csv
import shutil

SUBMETER_PROPOSTA_GLOBAL = False
CONFIRMAR_CHAMADA_GLOBAL = ""
GLOBAL_RUNTIME_DIR = Path("/content/falando_nela_g02_global_core")
GLOBAL_DRIVE_DIR = Path(
    "/content/drive/MyDrive/falando_nela/auditoria/"
    "pipeline_dados_v3/g02/schema-global-gpt56-20260724"
)

from pipeline_dados_v3.schema_normalizado import (
    GLOBAL_PROPOSAL_OPERATION_ID,
    GLOBAL_PROPOSAL_SCHEMA_VERSION,
    GPT56_GLOBAL_MAX_OUTPUT_TOKENS,
    GPT56_MAX_INPUT_TOKENS,
    estimate_gpt56_global_cost,
    global_proposal_json_schema,
    global_schema_prompt,
    sha256_file,
    sha256_json,
    sha256_text,
)

global_response_id = None
global_generation_input_tokens = None

def preserve_identical_file(source, destination):
    source = Path(source)
    destination = Path(destination)
    assert source.is_file(), f"Artefato ausente: {source}"
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert sha256_file(source) == sha256_file(destination), (
            f"Destino divergente; nada foi sobrescrito: {destination}"
        )
    else:
        shutil.copy2(source, destination)

if SUBMETER_PROPOSTA_GLOBAL:
    assert CONFIRMAR_CHAMADA_GLOBAL == GLOBAL_PROPOSAL_OPERATION_ID, (
        "Copie literalmente GLOBAL_PROPOSAL_OPERATION_ID para confirmar."
    )
    assert Path("/content/drive/MyDrive").is_dir(), "Monte o Drive."
    artifact_names = [
        "catalogo_global_gpt56.txt",
        "catalogo_global_crosswalk.csv",
        "catalogo_global_amostras.csv",
        "catalogo_global_manifest.json",
        "upload_token_count.json",
    ]
    for artifact_name in artifact_names:
        preserve_identical_file(
            GLOBAL_RUNTIME_DIR / artifact_name,
            GLOBAL_DRIVE_DIR / artifact_name,
        )

    upload_receipt = json.loads(
        (GLOBAL_RUNTIME_DIR / "upload_token_count.json").read_text(
            encoding="utf-8"
        )
    )
    catalog_path = GLOBAL_RUNTIME_DIR / "catalogo_global_gpt56.txt"
    assert upload_receipt["model"] == "gpt-5.6"
    assert upload_receipt["fits"] is True
    assert upload_receipt["catalog_sha256"] == sha256_file(catalog_path)

    if IN_COLAB:
        from google.colab import userdata

        api_key = userdata.get("OPENAI_API_KEY")
    else:
        api_key = os.environ.get("OPENAI_API_KEY")
    assert api_key, "OPENAI_API_KEY não está disponível."
    from openai import OpenAI

    client = OpenAI(api_key=api_key)
    global_model_input = [
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "file_id": upload_receipt["file_id"],
                },
                {"type": "input_text", "text": global_schema_prompt()},
            ],
        }
    ]
    global_text_config = {
        "format": {
            "type": "json_schema",
            "name": "falando_nela_global_schema_proposal",
            "strict": True,
            "schema": global_proposal_json_schema(),
        },
        "verbosity": "low",
    }
    exact_count = client.responses.input_tokens.count(
        model="gpt-5.6",
        input=global_model_input,
        reasoning={"effort": "medium"},
        text=global_text_config,
    )
    global_generation_input_tokens = exact_count.input_tokens
    assert global_generation_input_tokens <= GPT56_MAX_INPUT_TOKENS, (
        "A geração excede o limite conservador de entrada."
    )
    cost_low = estimate_gpt56_global_cost(
        input_tokens=global_generation_input_tokens,
        output_tokens=GPT56_GLOBAL_MAX_OUTPUT_TOKENS,
    )
    cost_high = estimate_gpt56_global_cost(
        input_tokens=global_generation_input_tokens,
        output_tokens=GPT56_GLOBAL_MAX_OUTPUT_TOKENS,
        cache_write_tokens=global_generation_input_tokens,
    )
    request_fingerprint = {
        "operation_id": GLOBAL_PROPOSAL_OPERATION_ID,
        "model": "gpt-5.6",
        "file_id": upload_receipt["file_id"],
        "catalog_sha256": upload_receipt["catalog_sha256"],
        "prompt_sha256": sha256_text(global_schema_prompt()),
        "schema_version": GLOBAL_PROPOSAL_SCHEMA_VERSION,
        "schema_sha256": sha256_json(global_proposal_json_schema()),
        "input_tokens": global_generation_input_tokens,
        "reasoning_effort": "medium",
        "max_output_tokens": GPT56_GLOBAL_MAX_OUTPUT_TOKENS,
        "truncation": "disabled",
    }
    request_sha256 = sha256_json(request_fingerprint)
    submission_path = GLOBAL_DRIVE_DIR / "submission_receipt.json"
    if submission_path.exists():
        submission = json.loads(submission_path.read_text(encoding="utf-8"))
        assert submission["request_sha256"] == request_sha256, (
            "Já existe submissão divergente; nenhuma nova chamada foi feita."
        )
        global_response_id = submission["response_id"]
        print("Submissão já existente; response_id reutilizado.")
    else:
        response = client.responses.create(
            model="gpt-5.6",
            input=global_model_input,
            reasoning={"effort": "medium"},
            text=global_text_config,
            max_output_tokens=GPT56_GLOBAL_MAX_OUTPUT_TOKENS,
            truncation="disabled",
            background=True,
            store=True,
            metadata={"operation_id": GLOBAL_PROPOSAL_OPERATION_ID},
        )
        global_response_id = response.id
        submission = {
            **request_fingerprint,
            "request_sha256": request_sha256,
            "response_id": global_response_id,
            "initial_status": response.status,
            "estimated_cost_usd_low": str(cost_low),
            "estimated_cost_usd_high": str(cost_high),
        }
        submission_path.write_text(
            json.dumps(submission, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        print("Chamada global submetida uma única vez.")
    print("response_id:", global_response_id)
    print("Tokens exatos com JSON Schema:", global_generation_input_tokens)
    print("Teto estimado (cache write + 32k output): US$", cost_high)
    print("Recibo preservado:", submission_path)
else:
    print("Submissão global bloqueada; nenhuma chamada paga foi feita.")

## 13. Consultar e preservar a resposta

Execute esta célula depois da submissão. Ela faz uma única consulta, portanto
não prende o runtime: se o estado ainda for `queued` ou `in_progress`, aguarde
e execute apenas esta mesma célula novamente. Ao concluir, ela preserva a
resposta bruta, valida o JSON e todos os `field_id` contra o crosswalk, grava
uso e custo real e mantém a proposta com estado `needs_human_review`. Nada é
aplicado ao schema nem aos dados.

In [ ]:
CONSULTAR_PROPOSTA_GLOBAL = False

if CONSULTAR_PROPOSTA_GLOBAL:
    submission_path = GLOBAL_DRIVE_DIR / "submission_receipt.json"
    assert submission_path.is_file(), "Submeta a chamada global primeiro."
    submission = json.loads(submission_path.read_text(encoding="utf-8"))
    if IN_COLAB:
        from google.colab import userdata

        api_key = userdata.get("OPENAI_API_KEY")
    else:
        api_key = os.environ.get("OPENAI_API_KEY")
    assert api_key, "OPENAI_API_KEY não está disponível."
    from openai import OpenAI
    from pipeline_dados_v3.schema_normalizado import (
        extract_output_text,
        extract_refusal,
        flatten_usage,
        model_dump,
        validate_global_proposal,
    )

    client = OpenAI(api_key=api_key)
    response = client.responses.retrieve(submission["response_id"])
    response_dict = model_dump(response)
    status_snapshot = {
        "operation_id": GLOBAL_PROPOSAL_OPERATION_ID,
        "response_id": response.id,
        "status": response.status,
        "error": response_dict.get("error"),
        "incomplete_details": response_dict.get("incomplete_details"),
    }
    (GLOBAL_DRIVE_DIR / "status_latest.json").write_text(
        json.dumps(status_snapshot, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print("Estado:", response.status)

    if response.status == "completed":
        raw_path = GLOBAL_DRIVE_DIR / "response_raw.json"
        raw_text = json.dumps(
            response_dict,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        ) + "\n"
        if raw_path.exists():
            assert raw_path.read_text(encoding="utf-8") == raw_text
        else:
            raw_path.write_text(raw_text, encoding="utf-8")

        refusal = extract_refusal(response_dict)
        assert not refusal, f"Resposta recusada: {refusal}"
        proposal = json.loads(extract_output_text(response, response_dict))
        with (GLOBAL_DRIVE_DIR / "catalogo_global_crosswalk.csv").open(
            encoding="utf-8", newline=""
        ) as handle:
            crosswalk_rows = list(csv.DictReader(handle))
        assert len(crosswalk_rows) == 23_786
        validate_global_proposal(proposal, crosswalk_rows)
        proposal_path = GLOBAL_DRIVE_DIR / "proposta_schema_global.json"
        proposal_text = json.dumps(
            proposal,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        ) + "\n"
        if proposal_path.exists():
            assert proposal_path.read_text(encoding="utf-8") == proposal_text
        else:
            proposal_path.write_text(proposal_text, encoding="utf-8")

        usage = flatten_usage(response_dict.get("usage") or {})
        actual_cost = estimate_gpt56_global_cost(
            input_tokens=usage["input_tokens"],
            cached_input_tokens=usage["cached_input_tokens"],
            cache_write_tokens=usage["cache_write_tokens"],
            output_tokens=usage["output_tokens"],
        )
        execution = {
            "operation_id": GLOBAL_PROPOSAL_OPERATION_ID,
            "scientific_gate": "needs_human_review",
            "proposal_applied": False,
            "response_id": response.id,
            "requested_model": "gpt-5.6",
            "resolved_model": response_dict.get("model", ""),
            "response_sha256": sha256_text(
                extract_output_text(response, response_dict)
            ),
            **usage,
            "actual_cost_usd": str(actual_cost),
            "pricing_as_of": "2026-07-24",
        }
        (GLOBAL_DRIVE_DIR / "execution.json").write_text(
            json.dumps(execution, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        print("Proposta validada e preservada:", proposal_path)
        print("Custo real estimado: US$", actual_cost)
        print("Gate científico: needs_human_review")
    elif response.status in {"queued", "in_progress"}:
        print("Ainda processando; reexecute somente esta célula mais tarde.")
    else:
        print("Estado terminal sem proposta; veja status_latest.json.")
else:
    print("Consulta bloqueada.")